# [CADWR ET DASHBOARD](https://dri-apps.projects.earthengine.app/view/cadwr-et-dashboard) 

# The code below prepares the assets and Org. logo for the EEapp above

## specify root path for the github repo

In [4]:
root_path = '/Users/blakeminor/Documents/GitHub/cadwr-basin-summaries'

gcloud_project_id = 'ee-bminor'

## import packages

In [5]:
import pandas as pd
import ee
import os
import json
import subprocess
import tempfile
from pathlib import PurePosixPath

# initialize the API
ee.Initialize(project=gcloud_project_id)


## pre-process tables to add columns for the EEapp

In [6]:
# read all tables
county_ag = pd.read_csv(os.path.join(root_path, 'csv_county_ag_lands', 'county_ag_lands_all_models.csv'))
county_all = pd.read_csv(os.path.join(root_path, 'csv_county_all_lands', 'county_all_lands_all_models.csv'))

gw_basin_ag = pd.read_csv(os.path.join(root_path, 'csv_gw_basin_ag_lands', 'gw_basin_ag_lands_all_models.csv'))
gw_basin_all = pd.read_csv(os.path.join(root_path, 'csv_gw_basin_all_lands', 'gw_basin_all_lands_all_models.csv'))

hydro_region_ag = pd.read_csv(os.path.join(root_path, 'csv_hydrologic_region_ag_lands', 'hydrologic_region_ag_lands_all_models.csv'))
hydro_region_all = pd.read_csv(os.path.join(root_path, 'csv_hydrologic_region_all_lands', 'hydrologic_region_all_lands_all_models.csv'))


# add new unique ID column "AGG_ID" (duplicate names, need to attach ID as well)
# add a flag column for missing months of ET data
county_ag['MISSING_ET_FLAG'] = (county_ag['PIXEL_COUNT'] == 0).astype(int)
county_ag['AGG_ID'] = county_ag['NAME'] + ' - ' + county_ag['GEOID'].astype(str)
county_all['MISSING_ET_FLAG'] = (county_all['PIXEL_COUNT'] == 0).astype(int)
county_all['AGG_ID'] = county_all['NAME'] + ' - ' + county_all['GEOID'].astype(str)

gw_basin_ag['MISSING_ET_FLAG'] = (gw_basin_ag['PIXEL_COUNT'] == 0).astype(int)
gw_basin_ag['AGG_ID'] = gw_basin_ag['Basin_Su_1'] + ' - ' + gw_basin_ag['Basin_Subb']
gw_basin_all['MISSING_ET_FLAG'] = (gw_basin_all['PIXEL_COUNT'] == 0).astype(int)
gw_basin_all['AGG_ID'] = gw_basin_all['Basin_Su_1'] + ' - ' + gw_basin_all['Basin_Subb']

hydro_region_ag['MISSING_ET_FLAG'] = (hydro_region_ag['PIXEL_COUNT'] == 0).astype(int)
hydro_region_ag['AGG_ID'] = hydro_region_ag['HR_NAME'] + ' - ' + hydro_region_ag['HR_ID'].astype(str)
hydro_region_all['MISSING_ET_FLAG'] = (hydro_region_all['PIXEL_COUNT'] == 0).astype(int)
hydro_region_all['AGG_ID'] = hydro_region_all['HR_NAME'] + ' - ' + hydro_region_all['HR_ID'].astype(str)

# build a dictionary of all dataframes to upload to cloud storage
full_data_dict = {
    'county_ag_lands_all_models': county_ag,
    'county_all_lands_all_models': county_all,
    'gw_basin_ag_lands_all_models': gw_basin_ag,
    'gw_basin_all_lands_all_models': gw_basin_all,
    'hydro_region_ag_lands_all_models': hydro_region_ag,
    'hydro_region_all_lands_all_models': hydro_region_all,
}

## upload ET summary tables to cloud storage

In [ ]:

for filename, df in full_data_dict.items():
    if not 'gw_basin' in filename:
        continue
    # fill nans in with 0s
    df = df.fillna(0)

    print(f'uploading {filename}.csv to cloud storage')

    # path to the bucket
    gcs_path = f'gs://openet/CDWR/CA_regions_ET_EEapp/{filename}.csv'

    # upload the dataframe to the bucket as a CSV
    df.to_csv(gcs_path, index=False)


## create GEE monthly assets from cloud bucket CSV tables

In [ ]:


# GCS wildcard
bucket_path = "gs://openet/CDWR/CA_regions_ET_EEapp/*.csv"

# Destination EE asset folder
asset_folder = (
    "projects/ee-bminor/assets/CDWR/ET_EEapp"
)

# List matching CSVs in GCS.
result = subprocess.run(
    ["gcloud", "storage", "ls", bucket_path],
    check=True,
    text=True,
    capture_output=True
)

cloud_files = [
    line.strip()
    for line in result.stdout.splitlines()
    if line.strip().endswith(".csv")
]

print(f"Found {len(cloud_files)} CSV file(s).")

for source_uri in cloud_files:

    filename = PurePosixPath(source_uri).name
    asset_name = PurePosixPath(filename).stem

    # Earth Engine asset/table name
    asset_id = f"{asset_folder}/{asset_name}"

    # Earth Engine table-upload manifest
    manifest = {
        "name": asset_id,
        "sources": [
            {
                "uris": [source_uri]
            }
        ]
    }

    # Use a temporary local file for the CLI manifest.
    with tempfile.NamedTemporaryFile(
        mode="w",
        suffix=".json",
        prefix=f"table_upload_manifest_{asset_name}_",
        delete=False
    ) as manifest_file:
        json.dump(manifest, manifest_file, indent=2)
        manifest_file_name = manifest_file.name

    try:
        print(f"Starting upload: {source_uri}")
        print(f"Target asset:    {asset_id}")

        subprocess.run(
            [
                "earthengine",
                "upload",
                "table",
                f"--manifest={manifest_file_name}",
            ],
            check=True,
            text=True
        )

    except subprocess.CalledProcessError as err:
        print(f"Upload failed for {source_uri}")
        print(err)

    finally:
        if os.path.exists(manifest_file_name):
            os.remove(manifest_file_name)

## create a feature view and feature collection to use in the EEapp
* needed to make a this fc with the updated unique ID, "AGG_ID" (NAME - ID)

In [ ]:

# set a unique ID property that contains the name and ID (ocassional duplicate names)
def setCountyID(ftr):
    return ftr.set('AGG_ID', ee.String(ftr.get('NAME')).cat(ee.String(' - ')).cat(ee.Number.parse(ftr.get('GEOID'))))

def setGwID(ftr):
    return ftr.set('AGG_ID', ee.String(ftr.get('Basin_Name')).cat(ee.String(' - ')).cat(ee.String(ftr.get('Basin_Subb'))))

def setHrID(ftr):
    return ftr.set('AGG_ID', ee.String(ftr.get('HR_NAME')).cat(ee.String(' - ')).cat(ee.Number.parse(ftr.get('HR_ID'))))


# feature collections with geometries
base_county_fc = (
    ee.FeatureCollection('projects/ee-cgmorton/assets/ca_counties')
        .select(['NAME', 'GEOID'])
        .map(setCountyID)
)
base_gw_basin_fc = (
    ee.FeatureCollection('projects/ee-cgmorton/assets/ca_gw_basins')
        .select(['Basin_Name', 'Basin_Subb'])
        .map(setGwID)
)
base_hr_region_fc = (
    ee.FeatureCollection('projects/ee-cgmorton/assets/ca_hydrologic_regions')
        .select(['HR_NAME', 'HR_ID'])
        .map(setHrID)
)

# aggregation dictionary with feature collections
agg_dict = {
    'county': base_county_fc,
    'gw_basin': base_gw_basin_fc,
    'hr_region': base_hr_region_fc,
}


for agg_type, fc in agg_dict.items():
    
    # FeatureView and FeatureCollection asset names
    out_asset_name_fc = f'{agg_type}_fc'
    out_asset_name_fv = f'{agg_type}_fv'

    # FeatureView and FeatureCollection assetIDs
    out_asset_id_fc = f'projects/ee-bminor/assets/CDWR/ET_EEapp/{out_asset_name_fc}'
    out_asset_id_fv = f'projects/ee-bminor/assets/CDWR/ET_EEapp/{out_asset_name_fv}'

    try:
        # FeatureCollection export task    
        out_task = ee.batch.Export.table.toAsset(**{
            'collection': fc,
            'description': f'cadwr_et_{agg_type}_aggregation_fc',
            'assetId': out_asset_id_fc,
        })
        out_task.start()
        
        # FeatureView export task
        out_task_view1 = ee.batch.Export.table.toFeatureView(**{
            'collection': fc,
            'description': f'cadwr_et_{agg_type}_aggregation_fv',
            'maxFeaturesPerTile': 10,
            'thinningStrategy': 'HIGHER_DENSITY',
            'thinningRanking': ['AGG_ID DESC', '.geometryType DESC', '.minZoomLevel ASC'],
            'zOrderRanking': 'AGG_ID', 
            'assetId': out_asset_id_fv,
            'selectors': ['AGG_ID']
        })
        out_task_view1.start()
    except Exception as e:
        print(e)
    
    print(f'tasks started for {agg_type} aggregation')

## create a DRI-CADWR logo for the EEapp

In [ ]:
from PIL import Image
import os

def combine_images(image_path1, image_path2, output_path, spacing=150):
    """Combines two images horizontally with specified spacing.

    Args:
        image_path1: Path to the first image.
        image_path2: Path to the second image.
        spacing: Spacing in pixels between the images.
        output_path: Path to save the combined image.
    """
    image1 = Image.open(image_path1).convert("RGBA")
    image2 = Image.open(image_path2).convert("RGBA")

    # image2 = image2.resize(image1.size)

    width1, height1 = image1.size
    width2, height2 = image2.size

    max_height = max(height1, height2)

    new_width = width1 + spacing + width2
    combined_image = Image.new("RGBA", (new_width, max_height))

    combined_image.paste(image1, (0, (max_height - height1) // 2))
    combined_image.paste(image2, (width1 + spacing, (max_height - height2) // 2))

    combined_image.save(output_path)
    print(f"Combined image saved to {output_path}")

if __name__ == "__main__":

    # input logos
    dri_logo = os.path.join(root_path, 'CADWR_ET_Dashboard', 'logos', 'official-dri-logo-white-bkgd.png')
    cadwr_logo = os.path.join(root_path, 'CADWR_ET_Dashboard', 'logos', 'dwr-logo-new.png')

    # output combined logo
    out_logo = os.path.join(root_path, 'CADWR_ET_Dashboard', 'logos', 'combined_dri_cadwr_logo.png')
    
    combine_images(dri_logo, cadwr_logo,  out_logo)